# Transformer-LSTM-SVM — GAN augmentation sweep (single seed: heavy model, MC disabled)

In [1]:
import pandas as pd, numpy as np, torch, torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)
import sys; sys.path.insert(0, "../")
from util import *
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

from models.transformer_lstm_svm import Transformer_LSTM_SVM
MODEL_NAME = "Transformer_LSTM_SVM"

SCENES = range(1, 7)          # scenes 1..6 (scene 0 = full data, no augmentation)
RATIOS = [0.0, 0.5, 1.0, 2.0] # 0.0 = no-augmentation baseline
SEEDS = [1]   # single seed — transformer training is too costly for 5x MC
EPOCHS, LR = 200, 1e-3


device: NVIDIA L4


In [2]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:, -1].astype("int64"))
    return X, y


def tensors_from_arrays(train_arr, val_arr, test_arr):
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train_arr, val_arr, test_arr)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def run_from_arrays(train_arr, val_arr, test_arr, device,
                    epochs=EPOCHS, lr=LR, seed=0):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = tensors_from_arrays(
        train_arr, val_arr, test_arr)
    model = Transformer_LSTM_SVM(n_classes=8, device=device, seed=seed)
    model.fit_backbone(X_tr, y_tr, X_val=X_va, y_val=y_va, epochs=epochs, lr=lr,
                       batch_size=50, seed=seed, verbose=False)
    model.fit_svm(X_tr, y_tr)
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision": p, "recall": r, "f1": f, "n_train": len(X_tr)}


In [3]:
from GANs.dcgans import train_class_gans, augment

val_arr  = pd.read_csv("../CSV_Files/eval/val.csv").to_numpy()
test_arr = pd.read_csv("../CSV_Files/eval/test.csv").to_numpy()

rows, gan_rows = [], []
for seed in SEEDS:
    for scene in SCENES:
        train_arr = pd.read_csv(f"../CSV_Files/seed{seed}/scene{scene}/train.csv").to_numpy()

        # Train the 8 per-class DCGANs ONCE per (seed, scene); reuse across ratios
        gans, gan_hist = train_class_gans(train_arr, seed=seed, device=device)
        for c, h in gan_hist.items():
            gan_rows.append({"seed": seed, "scene": scene, "class": c,
                             "d_acc_final":  round(h["d_acc"][-1], 4),
                             "d_loss_final": round(h["d_loss"][-1], 4),
                             "g_loss_final": round(h["g_loss"][-1], 4)})
        mean_dacc = float(np.mean([h["d_acc"][-1] for h in gan_hist.values()]))
        print(f">> seed {seed} scene {scene}: mean final D_acc = {mean_dacc:.3f}  "
              f"(near 0.5 = faithful, near 1.0 = generator collapsed)")


        for ratio in RATIOS:
            aug = augment(train_arr, gans, ratio, seed=seed, device=device)
            res = run_from_arrays(aug, val_arr, test_arr, device, seed=seed)
            res.update(seed=seed, scene=scene, ratio=ratio)
            rows.append(res)
            print(f"seed {seed} scene {scene} ratio {ratio}: "
                  f"F1={res['f1']:.4f} (n_train={res['n_train']})")

df = pd.DataFrame(rows)
df.to_csv(f"{MODEL_NAME.lower()}_gan_results_raw.csv", index=False)
pd.DataFrame(gan_rows).to_csv(f"{MODEL_NAME.lower()}_gan_diagnostics.csv", index=False)

agg = df.groupby(["scene", "ratio"])["f1"].agg(["mean", "std"]).reset_index()
print(agg.to_string(index=False))
agg.to_csv(f"{MODEL_NAME.lower()}_gan_results_agg.csv", index=False)


>> seed 1 scene 1: mean final D_acc = 0.607  (near 0.5 = faithful, near 1.0 = generator collapsed)


/home/ubuntu/miniconda3/envs/ad/lib/python3.10/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


seed 1 scene 1 ratio 0.0: F1=0.9650 (n_train=7866)


/home/ubuntu/miniconda3/envs/ad/lib/python3.10/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


seed 1 scene 1 ratio 0.5: F1=0.9713 (n_train=11790)


...

seed 1 scene 6 ratio 0.5: F1=0.3364 (n_train=118)


/home/ubuntu/miniconda3/envs/ad/lib/python3.10/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


seed 1 scene 6 ratio 1.0: F1=0.4323 (n_train=155)


/home/ubuntu/miniconda3/envs/ad/lib/python3.10/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


seed 1 scene 6 ratio 2.0: F1=0.4542 (n_train=232)
 scene  ratio     mean  std
     1    0.0 0.965002  NaN
     1    0.5 0.971294  NaN
     1    1.0 0.970610  NaN
     1    2.0 0.970030  NaN
     2    0.0 0.959376  NaN
     2    0.5 0.938961  NaN
     2    1.0 0.886973  NaN
     2    2.0 0.881921  NaN
     3    0.0 0.818136  NaN
     3    0.5 0.815070  NaN
     3    1.0 0.816602  NaN
     3    2.0 0.813186  NaN
     4    0.0 0.459380  NaN
     4    0.5 0.567081  NaN
     4    1.0 0.804518  NaN
     4    2.0 0.796798  NaN
     5    0.0 0.449345  NaN
     5    0.5 0.421155  NaN
     5    1.0 0.504652  NaN
     5    2.0 0.488092  NaN
     6    0.0 0.311583  NaN
     6    0.5 0.336377  NaN
     6    1.0 0.432310  NaN
     6    2.0 0.454155  NaN
